# 🔬 PHẦN 2: HỌC KHÔNG GIÁM SÁT (UNSUPERVISED ANOMALY DETECTION)
### Mục tiêu: Xây dựng hồ sơ hành vi mạng chuẩn từ `67.343` mẫu Normal & phát hiện dị biệt
---
### Hệ thống thuật toán & Công thức cốt lõi:
1. **PCA:** Nén không gian $122$ chiều giữ $\ge 90\%$ phương sai thông tin.
2. **Mini-Batch K-Means ($K=8$):** Tìm $8$ tâm hành vi mạng bình thường (Normal Centroids).
3. **DBSCAN ($\varepsilon=2.5, MinPts=5$):** Đo mật độ điểm, tự động phân lập nhiễu (`-1`).
4. **Khoảng cách Mahalanobis ($D_M$) & Euclidean ($D_E$):** Đo độ lệch khỏi tâm chuẩn, xác lập ngưỡng cắt phân vị thứ **95%**.


In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, DBSCAN

# Tái sử dụng pipeline tiền xử lý
BASE_DIR = Path("..").resolve()
TRAIN_FILE = BASE_DIR / "data" / "KDDTrain+.txt"
if not TRAIN_FILE.exists():
    TRAIN_FILE = Path(r"D:\data_miner\archive\KDDTrain+.txt")

COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']
SKEWED_COLS = ['duration', 'src_bytes', 'dst_bytes']
NUMERIC_COLS = [c for c in COLUMN_NAMES if c not in CATEGORICAL_COLS + ['label', 'difficulty']]

df = pd.read_csv(TRAIN_FILE, names=COLUMN_NAMES, header=None)
for c in CATEGORICAL_COLS:
    df[c] = df[c].astype(str).str.strip()
df['label'] = df['label'].astype(str).str.strip().str.lower()
normal_mask = (df['label'] == 'normal')

df_num = df[NUMERIC_COLS].copy()
for col in SKEWED_COLS:
    df_num[col] = np.log1p(np.maximum(0, df_num[col].astype(np.float32)))
scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_all = np.hstack([scaler.fit_transform(df_num.values), encoder.fit_transform(df[CATEGORICAL_COLS])]).astype(np.float32)

# CHỈ LẤY LƯU LƯỢNG CHUẨN (NORMAL) ĐỂ XÂY DỰNG BASELINE
X_normal = X_all[normal_mask]
print(f"Số lượng mẫu Normal dùng để tạo Baseline: {len(X_normal):,}")


### 1. Giảm chiều PCA và kiểm tra tỉ lệ phương sai giải thích tích lũy
- **Phân rã trị riêng trên ma trận hiệp phương sai của tập Normal:**
  $$\Sigma = \frac{1}{N} X_{\text{norm}}^T X_{\text{norm}}, \qquad \Sigma \mathbf{v}_k = \lambda_k \mathbf{v}_k$$
- **Chọn số thành phần chính $K$ sao cho bảo toàn $\ge 90\%$ tổng phương sai:**
  $$\frac{\sum_{k=1}^{K} \lambda_k}{\sum_{j=1}^{D} \lambda_j} \ge 0.90, \qquad X_{\text{PCA}} = X_{\text{norm}} W_K$$


In [2]:
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X_normal)
cum_var = np.cumsum(pca.explained_variance_ratio_) * 100

plt.figure(figsize=(8, 4), dpi=120)
plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o', color='#8e44ad', linewidth=2)
plt.axhline(90, color='red', linestyle='--', label='Ngưỡng bảo toàn thông tin 90%')
plt.title(f"Tỉ Lệ Phương Sai Tích Lũy Của PCA ({len(cum_var)} Thành Phần)", fontweight='bold')
plt.xlabel("Số thành phần chính (PC)")
plt.ylabel("Phương sai tích lũy (%)")
plt.legend()
plt.show()

print(f"✅ {len(cum_var)} thành phần chính nắm giữ: {cum_var[-1]:.2f}% tổng phương sai thông tin.")


### 2. Gom cụm Mini-Batch K-Means, DBSCAN & Khoảng cách Mahalanobis
- **Hàm mục tiêu Mini-Batch K-Means ($K = 8$):**
  $$J = \sum_{k=1}^{8} \sum_{\mathbf{x}_i \in S_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|_2^2 \to \min$$
- **Phân cụm mật độ DBSCAN ($\varepsilon = 2.5, MinPts = 5$):** Điểm $\mathbf{x}$ là điểm lõi nếu $|N_{\varepsilon}(\mathbf{x})| \ge 5$; các điểm ngoài biên mật độ bị gán nhãn `-1` (Noise).
- **Khoảng cách Euclidean ($D_E$) vs Khoảng cách Mahalanobis ($D_M$):**
  $$D_E(\mathbf{x}) = \min_k \|\mathbf{x}_{\text{PCA}} - \boldsymbol{\mu}_k\|_2, \qquad \Sigma_{\text{reg}} = \Sigma + 10^{-4}\mathbf{I}, \qquad D_M(\mathbf{x}) = \sqrt{(\mathbf{x}_{\text{PCA}} - \boldsymbol{\mu}_{k^*})^T \Sigma_{\text{reg}}^{-1} (\mathbf{x}_{\text{PCA}} - \boldsymbol{\mu}_{k^*})}$$
- **Ngưỡng bất thường bách phân vị thứ 95 ($\tau_{95}$):** $P(D_M \le \tau_{95} \mid \text{Normal}) = 0.95$.


In [3]:
kmeans = MiniBatchKMeans(n_clusters=8, batch_size=256, random_state=42, n_init="auto")
kmeans.fit(X_pca)
centroids = kmeans.cluster_centers_

# 1. Phân tích mật độ và điểm dị biệt bằng DBSCAN
dbscan = DBSCAN(eps=2.5, min_samples=5)
db_labels = dbscan.fit_predict(X_pca[:1500])
noise_ratio = (db_labels == -1).mean() * 100
print(f"✅ [DBSCAN] Tỉ lệ điểm biên/nhiễu phát hiện: {noise_ratio:.2f}%")

# 2. Tính khoảng cách Euclidean & Ngưỡng cắt
d_euc = cdist(X_pca, centroids, metric="euclidean").min(axis=1)
thresh_euc = float(np.percentile(d_euc, 95))

# 3. Tính ma trận hiệp phương sai nghịch đảo điều chuẩn & Mahalanobis
cov = np.cov(X_pca, rowvar=False) + np.eye(X_pca.shape[1], dtype=np.float32) * 1e-4
inv_cov = np.linalg.pinv(cov)

def get_mahalanobis(X_in):
    res = []
    for i in range(0, len(X_in), 5000):
        b = X_in[i:i+5000]
        df_b = b - centroids[cdist(b, centroids).argmin(axis=1)]
        res.append(np.sqrt(np.maximum(0, np.sum(df_b @ inv_cov * df_b, axis=1))))
    return np.concatenate(res)

d_mah = get_mahalanobis(X_pca)
thresh_mah = float(np.percentile(d_mah, 95))
print(f"✅ Ngưỡng cắt 95th Percentile: Mahalanobis = {thresh_mah:.2f} | Euclidean = {thresh_euc:.2f}")

# Tính khoảng cách Mahalanobis trên dữ liệu Tấn công thật trong KDDTrain+ để đối chiếu
X_pca_attack = pca.transform(X_all[~normal_mask])
d_mah_attack = get_mahalanobis(X_pca_attack)

# 4. Vẽ phân phối khoảng cách Mahalanobis thật (Normal vs Attack)
plt.figure(figsize=(10, 4.5), dpi=120)
sns.kdeplot(np.clip(d_mah, 0, 15), fill=True, color="royalblue", alpha=0.45, label="Lưu lượng chuẩn (Normal)")
sns.kdeplot(np.clip(d_mah_attack, 0, 15), fill=True, color="crimson", alpha=0.35, label="Lưu lượng tấn công (Attack)")
plt.axvline(thresh_mah, color="red", linestyle="--", linewidth=2, label=f"Ngưỡng cắt 95% ({thresh_mah:.2f})")
plt.title("Phân Phối Khoảng Cách Mahalanobis Trên Dữ Liệu Thật (Normal vs Attack)", fontweight="bold")
plt.xlabel("Khoảng cách Mahalanobis (D_M)")
plt.ylabel("Mật độ phân phối")
plt.xlim(0, 15)
plt.legend()
plt.show()
